# 🎯 목표: 0.6점 달성

## 현재 최고 성적
- **Optimized_GPTQ**: 0.5955 (9분 58초)

## 개선 전략
- **캘리브레이션 강화**: 384샘플, 768길이
- **기존 최적 파라미터 유지**: actorder=weight, dampening=0.001

In [11]:
import os
import torch
import shutil
import json

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.9.1
CUDA: False


In [12]:
# ============================================================================
# 🎯 0.6점 목표 설정
# ============================================================================

MODEL_ID = "./open/base_model"
OUT_DIR = "./model"
DATASET_ID = "LGAI-EXAONE/MANTA-1M"

# 캘리브레이션 설정 (강화)
NUM_SAMPLES = 384       # 256 → 384
MAX_SEQ_LEN = 768       # 512 → 768

# 양자화 설정
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE = ["embed_tokens", "lm_head"]  # vLLM 호환 필수!

# 최적화 파라미터 (검증된 설정)
BLOCK_SIZE = 128
DAMPENING = 0.001
ACTORDER = "weight"

print("=" * 60)
print("🎯 0.6점 목표 설정")
print("=" * 60)
print(f"캘리브레이션: {NUM_SAMPLES}샘플, {MAX_SEQ_LEN}길이")
print(f"actorder: {ACTORDER}")
print(f"dampening: {DAMPENING}")
print("=" * 60)

🎯 0.6점 목표 설정
캘리브레이션: 384샘플, 768길이
actorder: weight
dampening: 0.001


In [13]:
print("[1/5] 모델 로드...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32,
        trust_remote_code=True,
        device_map="cpu",
    )

print(f"  파라미터: {model.num_parameters():,}")

[1/5] 모델 로드...
  파라미터: 1,279,391,488


In [14]:
print(f"[2/5] 데이터셋 로드 ({NUM_SAMPLES}개)...")

ds = load_dataset(DATASET_ID, split=f"train[:{NUM_SAMPLES}]")

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)
print(f"  완료: {len(ds)}개")

[2/5] 데이터셋 로드 (384개)...
  완료: 384개


In [15]:
print("[3/5] GPTQ 양자화...")
print(f"  - actorder: {ACTORDER}")
print(f"  - dampening: {DAMPENING}")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        block_size=BLOCK_SIZE,
        dampening_frac=DAMPENING,
        actorder=ACTORDER,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQ_LEN,
    num_calibration_samples=NUM_SAMPLES,
)

print("  완료!")

[3/5] GPTQ 양자화...
  - actorder: weight
  - dampening: 0.001


Tokenizing:   0%|          | 0/384 [00:00<?, ? examples/s]

2026-02-12T09:40:31.314193+0900 | reset | INFO - Compression lifecycle reset
2026-02-12T09:40:31.315915+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-12T09:40:31.345251+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-12T09:40:31.345920+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-02-12T09:40:31.352171+0900 | dispatch_for_sequential | WARNING - CUDA/XPU is not available! Compressing model on CPU instead


W0212 09:40:31.393000 58885 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(1/31): Calibrating: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 384/384 [01:36<00:00,  3.99it/s]

2026-02-12T09:42:07.964846+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 384 samples


...

(30/31): Calibrating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 384/384 [01:34<00:00,  4.07it/s]

2026-02-12T10:41:12.719182+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 384 samples


2026-02-12T10:41:13.107889+0900 | compress | METRIC - time 0.39s
2026-02-12T10:41:13.108398+0900 | compress | METRIC - error 4665.19
2026-02-12T10:41:13.109690+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-12T10:41:13.109953+0900 | compress | METRIC - Compressed module size: 16.941056 MB
2026-02-12T10:41:13.111761+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 384 samples
2026-02-12T10:41:13.349147+0900 | compress | METRIC - time 0.24s
2026-02-12T10:41:13.349564+0900 | compress | METRIC - error 1320.42
2026-02-12T10:41:13.350506+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-12T10:41:13.350776+0900 | compress | METRIC - Compressed module size: 4.235264 MB
2026-02-12T10:41:13.351594+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 384 samples
2026-02-12T10:41:13.583871+0900 | compress | METRIC - time 0.23s
2026-02-12T10:

(31/31): Propagating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 384/384 [00:00<00:00, 2733.25it/s]


2026-02-12T10:41:38.509761+0900 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-12T10:41:38.517538+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
  완료!


In [16]:
print("[4/5] 모델 저장...")

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

total = sum(os.path.getsize(os.path.join(OUT_DIR, f)) for f in os.listdir(OUT_DIR))
print(f"  크기: {total/1e9:.2f} GB")

[4/5] 모델 저장...
2026-02-12T10:41:38.535839+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:02, 86.66it/s] 


  크기: 1.42 GB


In [17]:
# config.json 검증
with open(f"{OUT_DIR}/config.json") as f:
    cfg = json.load(f)

print("\n[검증] config.json")
print(f"  tie_word_embeddings: {cfg.get('tie_word_embeddings')}")

if cfg.get('tie_word_embeddings') == True:
    print("✅ vLLM 호환 확인!")
else:
    print("❌ 경고: tie_word_embeddings 문제!")


[검증] config.json
  tie_word_embeddings: True
✅ vLLM 호환 확인!


In [18]:
print("[5/5] 제출 파일 생성...")

zip_name = "submit"
if os.path.exists(f"{zip_name}.zip"):
    os.remove(f"{zip_name}.zip")

shutil.make_archive(zip_name, "zip", ".", OUT_DIR)

zip_size = os.path.getsize(f"{zip_name}.zip") / 1e9

print("\n" + "=" * 60)
print("🎯 제출 준비 완료!")
print("=" * 60)
print(f"파일: {zip_name}.zip ({zip_size:.2f} GB)")
print("=" * 60)

[5/5] 제출 파일 생성...

🎯 제출 준비 완료!
파일: submit.zip (0.88 GB)
